In [ ]:
import os, csv, datetime
from typing import List, Dict, Optional
import pandas as pd
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch  # 검색 툴
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()


In [ ]:
# -------------------------
# 1) 파일(CSV) 툴 2개
# -------------------------
@tool("csv_write", return_direct=False)
def csv_write(path: str, rows: List[Dict], mode: str = "a") -> str:
    """
    rows(list[dict])를 CSV 파일에 저장/추가한다.
    - path: csv 경로 (예: 'news.csv')
    - rows: 딕셔너리 목록. 키가 헤더가 됨. 새 파일이면 헤더를 자동 생성
    - mode: 'a' 추가(기본), 'w' 새로쓰기
    반환: 작성 행 수/헤더 정보
    """
    if not rows:
        return "rows가 비어있습니다."
    # 헤더 순서를 일정하게(가독성 좋은 기본 필드 우선)
    priority = ["date", "query", "title", "url", "source", "snippet"]
    keys = list({k for r in rows for k in r.keys()})
    ordered = priority + [k for k in keys if k not in priority]

    file_exists = os.path.isfile(path)
    write_header = (not file_exists) or (mode == "w")
    with open(path, mode, newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=ordered, extrasaction="ignore")
        if write_header:
            w.writeheader()
        for r in rows:
            w.writerow(r)
    return f"CSV 저장 완료: {path}, 행 수={len(rows)}, 헤더={ordered}"

@tool("csv_head", return_direct=False)
def csv_head(path: str, n: int = 5) -> str:
    """
    CSV 상위 n행 미리보기. (헤더 포함)
    """
    try:
        df = pd.read_csv(path)
        return df.head(n).to_string(index=False)
    except FileNotFoundError:
        return f"파일 없음: {path}"
    except Exception as e:
        return f"CSV 읽기 오류: {e}"



In [ ]:
# -------------------------
# 2) 검색 툴 (Tavily)
# -------------------------
tavily =

In [ ]:
# -------------------------
# 3) 프롬프트 & Agent
# -------------------------
SYSTEM_PROMPT = """너는 '웹검색'과 'CSV 저장' 두 가지 툴만 사용하는 어시스턴트다.
- 최신 뉴스나 웹결과는 Tavily 검색 툴로 가져온다.
- 사용자가 파일로 저장하라고 하면, 결과를 적절한 필드(date, query, title, url, source, snippet)로 정규화하여 csv_write를 호출한다.
- 파일 저장 후에는 csv_head로 상위 5행을 보여줘서 확인하게 한다.
- 한국어로 간결히 답하되, 마지막에 한 줄 결론을 덧붙여라.
"""




In [ ]:
print("\n=== 예시 A: 뉴스 요약만 ===")
result = 



In [ ]:
# (B) 검색 → CSV로 저장(한 번에)
print("\n=== 예시 B: 검색해서 news.csv로 저장 ===")
today = datetime.date.today().isoformat()
q = f"{today} AI 주요 뉴스 5개를 검색해서 news.csv로 저장해줘."
result = agent.invoke({"messages": [{"role": "user", "content": q}]})
print(result["messages"][-1].content)


In [ ]:
# (C) 이미 저장된 파일 미리보기
print("\n=== 예시 C: 저장 확인 ===")
result = agent.invoke({"messages": [{"role": "user", "content": "저장된 news.csv 상위 5행을 보여줘."}]})
print(result["messages"][-1].content)


[실습]
- 오늘 IT 대기업 인수 뉴스 5개 찾아서 it_news.csv로 저장하고 미리보기
- 엔비디아(또는 삼성전자) 관련 최신 기사 5개만 요약하고 nvidia_{오늘날짜}.csv로 저장
- 방금 만든 파일 상위 3행만 확인하기